# 1. Prepare GEX data

In [37]:
DROP_NULLS = True
DROP_LOUVEAU = True
SELECT_PRE_TREATMENT = True
SELECT_RNA_SEQ = False

## Some mappings

In [38]:
rna_seq_sources = ['Hugo et al.', 'Kwong et al.', 'Yan et al.']
q_pcr_sources = ['Louveau et al.']
micro_array_sources = ['Long et al.', 'Rizos et al.']

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.'
}

In [39]:
import polars as pl

gex = pl.read_csv("../dataset/original/gene_expressions.csv")
gex = gex.with_columns(pl.col('source').replace(source_map))
gex

id,creation_datetime,patientID,sample_id,HGNC,GeneID,description,value,temporality,source
i64,str,str,str,str,str,str,f64,str,str
1,"""2025-04-23 23:02:05.503260""","""LM_1""","""LMSAM_1""","""BRAF""",null,null,7.60798,"""pre treatment""","""Louveau et al."""
2,"""2025-04-23 23:02:05.503285""","""LM_1""","""LMSAM_1""","""RAF1""",null,null,16.095204,"""pre treatment""","""Louveau et al."""
3,"""2025-04-23 23:02:05.503298""","""LM_1""","""LMSAM_1""","""ARAF""",null,null,4.1515,"""pre treatment""","""Louveau et al."""
4,"""2025-04-23 23:02:05.503312""","""LM_1""","""LMSAM_1""","""PDGFRB""",null,null,1.199885,"""pre treatment""","""Louveau et al."""
5,"""2025-04-23 23:02:05.503324""","""LM_1""","""LMSAM_1""","""IGF1R""",null,null,5.47246,"""pre treatment""","""Louveau et al."""
…,…,…,…,…,…,…,…,…,…
8641387,"""2025-04-24 00:42:16.629746""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11A""",null,null,0.019542,"""progression""","""Hugo et al."""
8641388,"""2025-04-24 00:42:16.629757""","""HL_Shi-40""","""Pt21-DP2""","""ZYG11B""",null,null,4.04421,"""progression""","""Hugo et al."""
8641389,"""2025-04-24 00:42:16.629768""","""HL_Shi-40""","""Pt21-DP2""","""ZYX""",null,null,85.7967,"""progression""","""Hugo et al."""


## Select only 'pre-treatment', drop Louveau

In [40]:
if SELECT_PRE_TREATMENT == True:
    gex = gex.filter((pl.col('temporality') == 'pre treatment'))
if SELECT_RNA_SEQ == True:
    gex = gex.filter(pl.col('source').is_in(rna_seq_sources))
if DROP_LOUVEAU == True:
    gex = gex.filter(pl.col('source') != 'Louveau et al.')

## Drop useless features

In [41]:
gex = gex.drop(['id', 'creation_datetime', 'GeneID', 'description', 'temporality'])

## Sample is useless if patientID or HGNC is not given

In [42]:
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
92181,0,207000,0,0


In [43]:
if DROP_NULLS == True:
    gex = gex.drop_nulls(subset=['patientID', 'HGNC'])
    gex.select(pl.all().null_count())
    
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
0,0,0,0,0


## Add Method column

In [44]:
gex = gex.with_columns(
    pl.when(pl.col('source').is_in(rna_seq_sources))
    .then(pl.lit('RNA-seq'))
    .when(pl.col('source').is_in(micro_array_sources))
    .then(pl.lit('micro-array'))
    .otherwise(pl.lit('qPCR'))
    .alias('Method')
)
gex

patientID,sample_id,HGNC,value,source,Method
str,str,str,f64,str,str
"""YR_5306""","""03660445B""","""NAT2""",0.0,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""ADA""",26.233973,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""CDH2""",1.138609,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""AKT3""",12.692677,"""Yan et al.""","""RNA-seq"""
"""YR_5306""","""03660445B""","""GAGE12F""",0.0,"""Yan et al.""","""RNA-seq"""
…,…,…,…,…,…
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11A""",0.0625681,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11B""",5.74608,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYX""",45.907933,"""Hugo et al.""","""RNA-seq"""


## Remove duplicate (sample-gene) rows

In [45]:
dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (21_042, 6)
┌───────────┬───────────┬────────┬───────┬──────────────┬─────────┐
│ patientID ┆ sample_id ┆ HGNC   ┆ value ┆ source       ┆ Method  │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---          ┆ ---     │
│ str       ┆ str       ┆ str    ┆ f64   ┆ str          ┆ str     │
╞═══════════╪═══════════╪════════╪═══════╪══════════════╪═════════╡
│ KC_10     ┆ 10A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_10     ┆ 10A       ┆ ACE    ┆ 2.84  ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_12     ┆ 12A       ┆ ACE    ┆ 10.27 ┆ Kwong et al. ┆ RNA-seq │
│ KC_13     ┆ 13A       ┆ ACE    ┆ 18.67 ┆ Kwong et al. ┆ RNA-seq │
│ …         ┆ …         ┆ …      ┆ …     ┆ …            ┆ …       │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_7      ┆ 7A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir-95 ┆ 0.0   ┆ Kwong et al. ┆ RNA-seq │
│ KC_9      ┆ 9A        ┆ mir

In [46]:
gex = gex.sort('value', descending=True).unique(subset=['HGNC', 'sample_id'], keep='first')

dupes_pl = (
    gex
    .filter(pl.len().over(['HGNC', 'sample_id']) > 1)
    .sort(['HGNC', 'sample_id'])
)
print(dupes_pl)

shape: (0, 6)
┌───────────┬───────────┬──────┬───────┬────────┬────────┐
│ patientID ┆ sample_id ┆ HGNC ┆ value ┆ source ┆ Method │
│ ---       ┆ ---       ┆ ---  ┆ ---   ┆ ---    ┆ ---    │
│ str       ┆ str       ┆ str  ┆ f64   ┆ str    ┆ str    │
╞═══════════╪═══════════╪══════╪═══════╪════════╪════════╡
└───────────┴───────────┴──────┴───────┴────────┴────────┘


## Save pre-processed GEX

In [47]:
print(gex)
gex.write_csv(f'../dataset/created/gex.csv')

shape: (4_162_394, 6)
┌────────────┬─────────────────┬───────────┬───────────┬──────────────┬─────────────┐
│ patientID  ┆ sample_id       ┆ HGNC      ┆ value     ┆ source       ┆ Method      │
│ ---        ┆ ---             ┆ ---       ┆ ---       ┆ ---          ┆ ---         │
│ str        ┆ str             ┆ str       ┆ f64       ┆ str          ┆ str         │
╞════════════╪═════════════════╪═══════════╪═══════════╪══════════════╪═════════════╡
│ LR_MTP-009 ┆ 28094 PreB      ┆ BCLAF1    ┆ 1207.282  ┆ Long et al.  ┆ micro-array │
│ YR_2220    ┆ 05320216B       ┆ PLA2G4E   ┆ 0.036089  ┆ Yan et al.   ┆ RNA-seq     │
│ HL_Shi-15  ┆ Pt1-baseline    ┆ PPP1R9A   ┆ 0.395961  ┆ Hugo et al.  ┆ RNA-seq     │
│ LR_MTP-034 ┆ 28518_049E PreC ┆ LOC653232 ┆ 11346.65  ┆ Long et al.  ┆ micro-array │
│ KC_16      ┆ 16A             ┆ PMP22     ┆ 49.22     ┆ Kwong et al. ┆ RNA-seq     │
│ …          ┆ …               ┆ …         ┆ …         ┆ …            ┆ …           │
│ RL_WMD-007 ┆ 28067_010A PreB ┆

## Create GEX_MAT

In [48]:
gex_mat = gex.pivot(on='sample_id', index='HGNC', values='value')
gex_mat

HGNC,28094 PreB,05320216B,Pt1-baseline,28518_049E PreC,16A,Pt17-baseline,05420159C,05320093C,45534_085E PreB,46844 PreB,34A,Pt3-baseline,56241 PreB,05320399C,05320143B,7A,28067_010A PreB,56216 PreB,05320141B,Pt9-baseline,05320459B,12A,27473_049K PreB,05320328B,05320353C,05320372C,30230_035L PreB,05320011B,Pt6-baseline,27228_012A PreB,8766_022H PreC,56025 PreB,05420180C,28518_ 085G PreC,05320220B,29483_049G PreC,…,03660447B,2A,46225 PreC,05320232B,03660501B,30608 PreB,30509 PreB,2991 PreB,27551 PreB,30230_077K PreB,05320425B,24A,03660445B,04240151B,27473_010F PreB,05320381B,45534_072K PreB,15A,05320449B,05320243B,05420129C,03660555B,56460 PreB,05320385B,05320003B,05320384B,05320424B,Pt16-baseline,03660502B,8755_035H PreB,08626 PreC,28779 PreB,Pt10-baseline,05320119C,03660598B,45413 PreC,05320444B
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""BCLAF1""",1207.282,4.237369,30.4908,235.4447,17.26,18.1195,5.11878,6.423741,732.7158,872.6238,6.08,25.93115,1318.731,9.871449,11.439012,8.1,1353.935,1219.028,8.110015,39.78125,4.786404,6.66,303.0293,7.510313,6.786422,10.623197,896.9041,8.762577,11.951933,1322.013,1360.349,421.6427,6.462414,235.5909,5.585014,934.0073,…,6.829073,17.46,579.9982,7.281399,10.409055,811.1062,1090.873,1007.662,78.77097,960.1078,7.090598,31.89,11.62481,8.956245,351.5175,9.492155,678.5096,8.58,5.694362,6.973648,6.696623,7.048921,160.9126,6.774081,10.184101,8.328383,6.490579,9.734817,8.664979,313.6815,208.3033,1438.13,7.542365,7.327778,5.467553,575.8395,8.754171
"""PLA2G4E""",2.850116,0.036089,0.055628,4.696824,0.03,0.0,0.053727,0.0,9.305079,15.82763,0.0,5.57876,9.656116,0.005211,0.0,0.02,11.2462,5.183815,2.64166,0.036411,0.0,0.01,6.54293,0.004429,1.434829,0.0,8.719059,0.003113,0.0,14.36109,9.594628,7.432422,3.473698,6.920419,1.144128,3.81751,…,0.149581,0.03,12.91449,0.048546,9.262698,76.53121,6.846704,10.52048,10.98088,5.134191,14.186359,0.02,0.0,0.008376,3.091986,0.006817,8.655165,0.02,18.57811,0.0,0.014027,0.0,10.52198,7.094433,0.020034,2.271064,2.780736,0.0,0.486208,10.70527,10.7953,7.432281,6.176,6.09182,0.042667,8.587088,0.0
"""PPP1R9A""",22.14266,1.307164,0.395961,1.309181,3.87,0.255156,0.533329,5.020046,74.16772,77.88134,0.15,6.13674,4.101468,1.729816,10.924353,3.86,73.6711,25.93946,0.163391,11.44075,19.002026,0.08,39.20072,7.434152,0.371339,20.228643,37.53825,0.815195,0.813704,10.57327,24.42641,27.13193,0.089179,1.26496,0.114535,132.8442,…,4.167491,0.53,6.382381,12.898492,6.346381,16.27795,63.69662,7.25011,1.198996,70.38039,1.321882,0.19,9.450289,2.150999,45.533,0.74623,91.90913,0.19,0.146495,3.676745,0.336708,13.262008,41.54336,8.587068,23.077525,0.5375287,1.185453,0.822773,2.330955,34.06139,79.35284,2.165423,0.4676475,2.988701,4.596764,52.05827,25.08646
"""LOC653232""",11541.6,null,null,11346.65,null,null,null,null,13904.06,13175.09,null,null,10282.96,null,null,null,15803.08,9971.659,null,null,null,null,11249.7,null,null,null,8108.061,null,null,15623.48,8619.708,8659.229,null,13743.97,null,13827.05,…,null,null,14442.91,null,null,8493.185,12875.86,12429.14,13737.17,9775.581,null,null,null,null,11695.24,null,13021.29,null,null,null,null,null,12204.89,null,null,null,null,null,null,14158.83,14946.39,13764.39,null,null,null,8720.374,null
"""PMP22""",3411.054,54.578361,151.8825,3891.525,49.22,213.541667,61.882859,51.377805,1559.453,6049.883,123.81,185.3815,3478.619,23.193317,39.119683,120.05,3023.66,2362.372,90.680685,56.1284,41.867883,28.95,2019.354,81.06913,120.437783,32.46112,1110.531,119.302463,64.096267,2616.299,1685.458,6348.463,55.751287,3582.444,60.065408,2347.284,…,64.651308,105.69,2723.516,110.249553,46.114725,4550.212,3162.355,4960.495,2906.208,1109.989,45.47774,98.65,34.026404,120.198641,2064.2,33.545861,1718.761,37.1,88.893247,77.518808,106.374506,5

In [49]:
gex_mat.write_csv(f'../dataset/created/gex_mat.csv')